In [12]:
class EmailAssistant:
    
    def generate_email(self, intent, facts, tone):
        
        subject = f"Subject: {intent.title()}"
        
        if "interview" in intent.lower():
            greeting = "Dear Hiring Manager,"
        else:
            greeting = "Dear Sir/Madam,"
        
        facts_text = " ".join([fact.capitalize() + "." for fact in facts])
        intent_lower = intent.lower()
        
        if "apology" in intent_lower:
            body = f"""
I sincerely apologize for the inconvenience caused.

{facts_text}

I take full responsibility and will ensure this does not happen again.
"""
        
        elif "follow" in intent_lower:
            body = f"""
I hope you are doing well.

I am writing to follow up regarding my recent interaction.

{facts_text}

I would appreciate any updates at your convenience.
"""
        
        elif "leave" in intent_lower:
            body = f"""
I would like to request leave for the following reason.

{facts_text}

Kindly grant approval for the same.
"""
        
        elif "meeting" in intent_lower:
            body = f"""
This is a reminder regarding the upcoming meeting.

{facts_text}

Please ensure your availability.
"""
        
        elif "request" in intent_lower:
            body = f"""
I am writing to request the following information.

{facts_text}

I would appreciate your response at the earliest.
"""
        
        else:
            body = f"""
I am writing regarding the matter.

{facts_text}

Please let me know if further information is required.
"""
        
        closing = """
Looking forward to your response.

Best regards,  
[Your Name]
"""
        
        email = f"{subject}\n\n{greeting}\n{body}\n{closing}"
        
        return {"email": email.strip()}

In [13]:
class BasicModel:
    def generate_email(self, intent, facts, tone):
        return {
            "email": f"Subject: {intent}\n\n" + " ".join(facts)
        }

In [14]:
assistant = EmailAssistant()
basic_model = BasicModel()

In [15]:
test_cases = [
    {
        "intent": "Follow up after interview",
        "facts": ["Interview on Monday", "Waiting for feedback"],
        "tone": "Professional",
        "human_email": "Follow-up email asking for feedback politely"
    },
    {
        "intent": "Leave request",
        "facts": ["Medical reason", "3 days leave"],
        "tone": "Formal",
        "human_email": "Formal leave request"
    },
    {
        "intent": "Meeting reminder",
        "facts": ["Meeting tomorrow", "10 AM"],
        "tone": "Professional",
        "human_email": "Reminder email"
    },
    {
        "intent": "Apology for delay",
        "facts": ["Server issue caused delay"],
        "tone": "Empathetic",
        "human_email": "Apology email"
    },
    {
        "intent": "Information request",
        "facts": ["Need pricing", "Timeline"],
        "tone": "Formal",
        "human_email": "Request email"
    },
    {
        "intent": "Project update",
        "facts": ["Phase 1 completed", "Phase 2 ongoing"],
        "tone": "Professional",
        "human_email": "Update email"
    },
    {
        "intent": "Client follow-up",
        "facts": ["Proposal sent", "Awaiting reply"],
        "tone": "Professional",
        "human_email": "Follow-up email"
    },
    {
        "intent": "Team appreciation",
        "facts": ["Great teamwork", "Project success"],
        "tone": "Appreciative",
        "human_email": "Appreciation email"
    },
    {
        "intent": "Leave extension",
        "facts": ["Need 2 more days", "Recovery ongoing"],
        "tone": "Formal",
        "human_email": "Extension email"
    },
    {
        "intent": "Job application",
        "facts": ["Applied last week", "Interested role"],
        "tone": "Professional",
        "human_email": "Application email"
    }
]

In [16]:
def fact_score(email, facts):
    return sum(1 for f in facts if f.lower() in email.lower()) / len(facts)


def tone_score(email, tone):
    email = email.lower()
    if tone == "Empathetic" and ("sorry" in email or "apologize" in email):
        return 1
    elif tone == "Professional" and "regards" in email:
        return 1
    elif tone == "Formal" and "dear" in email:
        return 1
    else:
        return 0


def structure_score(email):
    score = 0
    if "subject" in email.lower():
        score += 1
    if "dear" in email.lower():
        score += 1
    if "regards" in email.lower():
        score += 1
    return score / 3

In [17]:
results = []

for test in test_cases:
    
    A = assistant.generate_email(test["intent"], test["facts"], test["tone"])["email"]
    B = basic_model.generate_email(test["intent"], test["facts"], test["tone"])["email"]
    
    results.append({
        "intent": test["intent"],
        "A_fact": fact_score(A, test["facts"]),
        "A_tone": tone_score(A, test["tone"]),
        "A_structure": structure_score(A),
        "B_fact": fact_score(B, test["facts"]),
        "B_tone": tone_score(B, test["tone"]),
        "B_structure": structure_score(B)
    })

In [18]:
import json

with open("results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Saved results.json")

Saved results.json


In [19]:
def avg(key):
    return sum(r[key] for r in results) / len(results)

print("Model A:")
print("Fact:", avg("A_fact"))
print("Tone:", avg("A_tone"))
print("Structure:", avg("A_structure"))

print("\nModel B:")
print("Fact:", avg("B_fact"))
print("Tone:", avg("B_tone"))
print("Structure:", avg("B_structure"))

Model A:
Fact: 1.0
Tone: 0.9
Structure: 1.0

Model B:
Fact: 1.0
Tone: 0.0
Structure: 0.33333333333333337


In [20]:
def save_email(text, filename="email.txt"):
    with open(filename, "w") as f:
        f.write(text)
    return filename

sample = assistant.generate_email(
    "Follow up after interview",
    ["Interview completed on Monday", "Waiting for feedback"],
    "Professional"
)

file = save_email(sample["email"])
print("Saved:", file)

Saved: email.txt


In [21]:
print("Metric 1: Fact Score → checks if all facts are included")
print("Metric 2: Tone Score → checks tone correctness")
print("Metric 3: Structure Score → checks subject, greeting, closing")

Metric 1: Fact Score → checks if all facts are included
Metric 2: Tone Score → checks tone correctness
Metric 3: Structure Score → checks subject, greeting, closing


In [23]:
print("\n===== FULL EVALUATION RESULTS =====\n")

for r in results:
    print(r)


===== FULL EVALUATION RESULTS =====

{'intent': 'Follow up after interview', 'A_fact': 1.0, 'A_tone': 1, 'A_structure': 1.0, 'B_fact': 1.0, 'B_tone': 0, 'B_structure': 0.3333333333333333}
{'intent': 'Leave request', 'A_fact': 1.0, 'A_tone': 1, 'A_structure': 1.0, 'B_fact': 1.0, 'B_tone': 0, 'B_structure': 0.3333333333333333}
{'intent': 'Meeting reminder', 'A_fact': 1.0, 'A_tone': 1, 'A_structure': 1.0, 'B_fact': 1.0, 'B_tone': 0, 'B_structure': 0.3333333333333333}
{'intent': 'Apology for delay', 'A_fact': 1.0, 'A_tone': 1, 'A_structure': 1.0, 'B_fact': 1.0, 'B_tone': 0, 'B_structure': 0.3333333333333333}
{'intent': 'Information request', 'A_fact': 1.0, 'A_tone': 1, 'A_structure': 1.0, 'B_fact': 1.0, 'B_tone': 0, 'B_structure': 0.3333333333333333}
{'intent': 'Project update', 'A_fact': 1.0, 'A_tone': 1, 'A_structure': 1.0, 'B_fact': 1.0, 'B_tone': 0, 'B_structure': 0.3333333333333333}
{'intent': 'Client follow-up', 'A_fact': 1.0, 'A_tone': 1, 'A_structure': 1.0, 'B_fact': 1.0, 'B_tone'

In [24]:
print("\n===== FINAL ANALYSIS =====\n")

print("Model A performs better than Model B.")
print("Reason: Model A generates structured, professional emails with correct tone.")
print("Model B lacks formatting, tone, and readability.")

print("\nRecommendation:")
print("Model A should be used for production due to higher quality outputs.")


===== FINAL ANALYSIS =====

Model A performs better than Model B.
Reason: Model A generates structured, professional emails with correct tone.
Model B lacks formatting, tone, and readability.

Recommendation:
Model A should be used for production due to higher quality outputs.
